# Custom Standards — Define & Map to Your Own Data Model

This notebook shows how to:
1. Define a custom clinical data standard as a YAML file
2. Load it into Portiere
3. Run the schema + concept mapping pipeline against your custom standard
4. Export and review the results

Use this when your target is a hospital-internal CDM, a national registry, a research data warehouse, or any custom schema.

In [1]:
import sys
sys.path.insert(0, "../../packages/sdk/src")

## Step 1 — Write Your YAML Standard Definition

A standard YAML file defines:
- **Metadata** — name, version, organization
- **Entities** — tables in your target schema
- **Fields** — columns with types, descriptions, and constraints
- **Source patterns** — column-name patterns for fast matching (high confidence)
- **Embedding descriptions** — AI-friendly text for semantic matching (when patterns don't match)

In [2]:
import os, tempfile

CUSTOM_YAML = """
name: "hospital_cdm_v1"
version: "1.0"
standard_type: "relational"
organization: "General Hospital Research"
description: "Internal clinical data model for General Hospital"

# Default fallback for unmapped columns
default_entity: "clinical_notes"
default_field: "note_text"

entities:
  patients:
    description: "Core patient demographics"
    fields:
      patient_id:
        type: integer
        required: true
        description: "Unique patient identifier"
        ddl: "INTEGER PRIMARY KEY"
      mrn:
        type: string
        description: "Medical record number"
        ddl: "VARCHAR(20) NOT NULL"
      first_name:
        type: string
        description: "Patient first name"
        ddl: "VARCHAR(100)"
      last_name:
        type: string
        description: "Patient last name"
        ddl: "VARCHAR(100)"
      date_of_birth:
        type: date
        description: "Patient date of birth"
        ddl: "DATE NOT NULL"
      sex:
        type: string
        description: "Biological sex (M/F/U)"
        ddl: "VARCHAR(1) NOT NULL"
      race:
        type: string
        description: "Patient race/ethnicity"
        ddl: "VARCHAR(50)"
      zip_code:
        type: string
        description: "Patient ZIP code for geocoding"
        ddl: "VARCHAR(10)"

    source_patterns:
      patient_id: ["patients", "patient_id"]
      subject_id: ["patients", "patient_id"]
      mrn: ["patients", "mrn"]
      medical_record: ["patients", "mrn"]
      first_name: ["patients", "first_name"]
      last_name: ["patients", "last_name"]
      dob: ["patients", "date_of_birth"]
      birth_date: ["patients", "date_of_birth"]
      gender: ["patients", "sex"]
      sex: ["patients", "sex"]
      race: ["patients", "race"]
      ethnicity: ["patients", "race"]
      zip: ["patients", "zip_code"]
      postal_code: ["patients", "zip_code"]

    embedding_descriptions:
      patient_id: "unique patient identifier number"
      mrn: "medical record number hospital identifier"
      first_name: "patient given first name"
      last_name: "patient family surname"
      date_of_birth: "patient birth date birthday"
      sex: "biological sex gender male female"
      race: "patient race ethnicity demographic"
      zip_code: "patient zip postal code address location"

  encounters:
    description: "Hospital visits and admissions"
    fields:
      encounter_id:
        type: integer
        required: true
        description: "Unique encounter identifier"
        ddl: "INTEGER PRIMARY KEY"
      patient_id:
        type: integer
        required: true
        fk: "patients.patient_id"
        description: "Reference to patients table"
        ddl: "INTEGER NOT NULL"
      admit_date:
        type: datetime
        description: "Admission date and time"
        ddl: "TIMESTAMP NOT NULL"
      discharge_date:
        type: datetime
        description: "Discharge date and time"
        ddl: "TIMESTAMP"
      encounter_type:
        type: string
        description: "Type of encounter (inpatient, outpatient, ED, telehealth)"
        ddl: "VARCHAR(20) NOT NULL"
      department:
        type: string
        description: "Admitting department or unit"
        ddl: "VARCHAR(100)"

    source_patterns:
      encounter_id: ["encounters", "encounter_id"]
      visit_id: ["encounters", "encounter_id"]
      hadm_id: ["encounters", "encounter_id"]
      admission_id: ["encounters", "encounter_id"]
      admit_date: ["encounters", "admit_date"]
      admittime: ["encounters", "admit_date"]
      discharge_date: ["encounters", "discharge_date"]
      dischtime: ["encounters", "discharge_date"]
      encounter_type: ["encounters", "encounter_type"]
      visit_type: ["encounters", "encounter_type"]
      department: ["encounters", "department"]
      unit: ["encounters", "department"]

    embedding_descriptions:
      encounter_id: "hospital encounter visit admission identifier"
      patient_id: "patient reference foreign key for encounter"
      admit_date: "admission date time when patient was admitted"
      discharge_date: "discharge date time when patient left hospital"
      encounter_type: "visit type inpatient outpatient emergency"
      department: "hospital department unit ward admitting"

  diagnoses:
    description: "Diagnosis records linked to encounters"
    fields:
      diagnosis_id:
        type: integer
        required: true
        description: "Unique diagnosis record identifier"
        ddl: "INTEGER PRIMARY KEY"
      encounter_id:
        type: integer
        required: true
        fk: "encounters.encounter_id"
        description: "Reference to encounters table"
        ddl: "INTEGER NOT NULL"
      icd_code:
        type: string
        required: true
        description: "ICD-10-CM diagnosis code"
        ddl: "VARCHAR(10) NOT NULL"
      description:
        type: string
        description: "Human-readable diagnosis description"
        ddl: "VARCHAR(500)"
      diagnosis_type:
        type: string
        description: "Primary, secondary, or admitting diagnosis"
        ddl: "VARCHAR(20)"

    source_patterns:
      icd_code: ["diagnoses", "icd_code"]
      icd9_code: ["diagnoses", "icd_code"]
      icd10_code: ["diagnoses", "icd_code"]
      diagnosis_code: ["diagnoses", "icd_code"]
      dx_code: ["diagnoses", "icd_code"]
      diagnosis_description: ["diagnoses", "description"]
      long_title: ["diagnoses", "description"]

    embedding_descriptions:
      diagnosis_id: "diagnosis record unique identifier"
      encounter_id: "encounter visit reference for diagnosis"
      icd_code: "ICD diagnosis code ICD-10-CM ICD-9"
      description: "diagnosis description name condition text"
      diagnosis_type: "primary secondary admitting diagnosis type"

  clinical_notes:
    description: "Free-text clinical documentation"
    fields:
      note_id:
        type: integer
        required: true
        description: "Unique note identifier"
        ddl: "INTEGER PRIMARY KEY"
      encounter_id:
        type: integer
        fk: "encounters.encounter_id"
        description: "Reference to encounters table"
        ddl: "INTEGER NOT NULL"
      note_type:
        type: string
        required: true
        description: "Note category (discharge summary, progress note, consult)"
        ddl: "VARCHAR(50) NOT NULL"
      note_text:
        type: text
        required: true
        description: "Full text content of the clinical note"
        ddl: "TEXT NOT NULL"
      author:
        type: string
        description: "Note author (physician, nurse, etc.)"
        ddl: "VARCHAR(200)"

    source_patterns:
      note_text: ["clinical_notes", "note_text"]
      clinical_note: ["clinical_notes", "note_text"]
      note_type: ["clinical_notes", "note_type"]
      category: ["clinical_notes", "note_type"]

    embedding_descriptions:
      note_id: "clinical note unique identifier"
      encounter_id: "encounter visit reference for note"
      note_type: "note category type discharge summary progress consult"
      note_text: "clinical note full text content documentation"
      author: "note author physician nurse provider writer"
"""

# Write YAML to a temp file
custom_yaml_path = os.path.join(tempfile.gettempdir(), "hospital_cdm_v1.yaml")
with open(custom_yaml_path, "w") as f:
    f.write(CUSTOM_YAML)

print(f"Custom standard written to: {custom_yaml_path}")

Custom standard written to: /var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/hospital_cdm_v1.yaml


## Step 2 — Load and Inspect the Custom Standard

Two ways to load a custom YAML standard:

| Method | When to use |
|--------|-------------|
| `get_target_model("custom:/path/to/file.yaml")` | Factory function, works with `portiere.init()` |
| `YAMLTargetModel.from_file("/path/to/file.yaml")` | Direct loading for inspection |

In [3]:
from portiere.models.target_model import get_target_model
from portiere.standards import YAMLTargetModel

# Option 1: custom: prefix (recommended for pipelines)
model = get_target_model(f"custom:{custom_yaml_path}")

# Option 2: direct loading
# model = YAMLTargetModel.from_file(custom_yaml_path)

print(f"Name: {model.name}")
print(f"Version: {model.version}")
print(f"Type: {model.standard_type}")
print(f"Organization: {model.organization}")

Name: hospital_cdm_v1
Version: 1.0
Type: relational
Organization: General Hospital Research


In [4]:
# Inspect entities and fields
schema = model.get_schema()
total_fields = sum(len(fields) for fields in schema.values())
print(f"{len(schema)} entities, {total_fields} total fields:\n")

for entity, fields in schema.items():
    print(f"  {entity}: {len(fields)} fields — {', '.join(fields)}")

4 entities, 24 total fields:

  patients: 8 fields — patient_id, mrn, first_name, last_name, date_of_birth, sex, race, zip_code
  encounters: 6 fields — encounter_id, patient_id, admit_date, discharge_date, encounter_type, department
  diagnoses: 5 fields — diagnosis_id, encounter_id, icd_code, description, diagnosis_type
  clinical_notes: 5 fields — note_id, encounter_id, note_type, note_text, author


In [5]:
# Inspect source patterns (fast-path matching)
patterns = model.get_source_patterns()
print(f"{len(patterns)} source patterns:\n")
for pattern, (entity, field) in sorted(patterns.items()):
    print(f"  '{pattern}' → {entity}.{field}")

37 source patterns:

  'admission_id' → encounters.['encounters', 'encounter_id']
  'admit_date' → encounters.['encounters', 'admit_date']
  'admittime' → encounters.['encounters', 'admit_date']
  'birth_date' → patients.['patients', 'date_of_birth']
  'category' → clinical_notes.['clinical_notes', 'note_type']
  'clinical_note' → clinical_notes.['clinical_notes', 'note_text']
  'department' → encounters.['encounters', 'department']
  'diagnosis_code' → diagnoses.['diagnoses', 'icd_code']
  'diagnosis_description' → diagnoses.['diagnoses', 'description']
  'discharge_date' → encounters.['encounters', 'discharge_date']
  'dischtime' → encounters.['encounters', 'discharge_date']
  'dob' → patients.['patients', 'date_of_birth']
  'dx_code' → diagnoses.['diagnoses', 'icd_code']
  'encounter_id' → encounters.['encounters', 'encounter_id']
  'encounter_type' → encounters.['encounters', 'encounter_type']
  'ethnicity' → patients.['patients', 'race']
  'first_name' → patients.['patients', 'fir

In [6]:
# Inspect embedding descriptions (AI matching)
descriptions = model.get_target_descriptions_tupled()
print(f"{len(descriptions)} embedding descriptions:\n")
for (entity, field), desc in descriptions.items():
    print(f"  {entity}.{field}: {desc}")

24 embedding descriptions:

  patients.patient_id: unique patient identifier number
  patients.mrn: medical record number hospital identifier
  patients.first_name: patient given first name
  patients.last_name: patient family surname
  patients.date_of_birth: patient birth date birthday
  patients.sex: biological sex gender male female
  patients.race: patient race ethnicity demographic
  patients.zip_code: patient zip postal code address location
  encounters.encounter_id: hospital encounter visit admission identifier
  encounters.patient_id: patient reference foreign key for encounter
  encounters.admit_date: admission date time when patient was admitted
  encounters.discharge_date: discharge date time when patient left hospital
  encounters.encounter_type: visit type inpatient outpatient emergency
  encounters.department: hospital department unit ward admitting
  diagnoses.diagnosis_id: diagnosis record unique identifier
  diagnoses.encounter_id: encounter visit reference for diagn

In [7]:
# Generate DDL for the custom standard
ddl = model.generate_ddl()
print(ddl)

CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY,
    mrn VARCHAR(20) NOT NULL,
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    date_of_birth DATE NOT NULL,
    sex VARCHAR(1) NOT NULL,
    race VARCHAR(50),
    zip_code VARCHAR(10)
);

CREATE TABLE encounters (
    encounter_id INTEGER PRIMARY KEY,
    patient_id INTEGER NOT NULL,
    admit_date TIMESTAMP NOT NULL,
    discharge_date TIMESTAMP,
    encounter_type VARCHAR(20) NOT NULL,
    department VARCHAR(100),
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
);

CREATE TABLE diagnoses (
    diagnosis_id INTEGER PRIMARY KEY,
    encounter_id INTEGER NOT NULL,
    icd_code VARCHAR(10) NOT NULL,
    description VARCHAR(500),
    diagnosis_type VARCHAR(20),
    FOREIGN KEY (encounter_id) REFERENCES encounters(encounter_id)
);

CREATE TABLE clinical_notes (
    note_id INTEGER PRIMARY KEY,
    encounter_id INTEGER NOT NULL,
    note_type VARCHAR(50) NOT NULL,
    note_text TEXT NOT NULL,
    author VAR

## Step 3 — Run the Pipeline with Your Custom Standard

Use the `custom:` prefix in `target_model` to run the full pipeline —
schema mapping, concept mapping, and ETL.

In [8]:
import portiere

project = portiere.init(
    name="MIMIC → Hospital CDM",
    target_model=f"custom:{custom_yaml_path}",
    vocabularies=["ICD10CM", "SNOMED"],
)

print(f"Project: {project.name}")
print(f"Target: {project.target_model}")
print(f"Engine: {project.engine.engine_name}")

2026-04-16 00:35:42 [info     ] PolarsEngine initialized      


2026-04-16 00:35:42 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project: MIMIC → Hospital CDM
Target: custom:/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/hospital_cdm_v1.yaml
Engine: polars


In [9]:
# Add source data (e.g., a MIMIC-style CSV)
source = project.add_source("example_assets/15_custom_standards/patients.csv", name="patients")
print(f"Source: {source['name']}, {len(source.get('columns', []))} columns")
print(f"Columns: {source.get('columns', [])}")


2026-04-16 00:35:42 [info     ] project.source_added           project='MIMIC → Hospital CDM' source=patients


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:35:45 [info     ] gx_profiler.profiled           columns=6 rows=5 source=patients


2026-04-16 00:35:45 [info     ] project.profiled               source=patients


Source: patients, 6 columns
Columns: ['patient_id', 'full_name', 'date_of_birth', 'sex', 'medical_record_number', 'insurance_id']


In [10]:
# Schema mapping — maps source columns to your custom standard's entities/fields
schema_map = project.map_schema(source)

print(f"Schema mapping: {len(schema_map.items)} columns mapped\n")
for item in schema_map.items:
    print(f"  {item.source_column} → {item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:35:45 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=custom:/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/hospital_cdm_v1.yaml


2026-04-16 00:35:45 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:35:45 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:35:45 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:35:47 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:35:47 [info     ] Stage 2 complete               auto=3 review=0 total=6


2026-04-16 00:35:47 [info     ] local_storage.schema_mapping_saved items_count=6 project='MIMIC → Hospital CDM'


2026-04-16 00:35:47 [info     ] project.schema_mapped          auto=3 project='MIMIC → Hospital CDM' total=6


Schema mapping: 6 columns mapped

  patient_id → person.person_id (confidence=0.95)
  full_name → observation.observation_source_value (confidence=0.30)
  date_of_birth → person.birth_datetime (confidence=0.95)
  sex → person.gender_concept_id (confidence=0.95)
  medical_record_number → observation.observation_source_value (confidence=0.30)
  insurance_id → observation.observation_source_value (confidence=0.30)


In [11]:
# Concept mapping — maps source values to standard vocabularies
concept_map = project.map_concepts(source=source)

print(f"Concept mapping: {len(concept_map.items)} terms mapped")
summary = concept_map.summary()
print(f"  Auto-mapped: {summary['auto_mapped']}")
print(f"  Needs review: {summary['needs_review']}")
print(f"  Manual required: {summary['manual_required']}")

2026-04-16 00:35:47 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=None


2026-04-16 00:35:47 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=0


2026-04-16 00:35:47 [info     ] local_storage.concept_mapping_saved items_count=0 project='MIMIC → Hospital CDM'


2026-04-16 00:35:47 [info     ] project.concepts_mapped        auto_rate=0 project='MIMIC → Hospital CDM' total=0


Concept mapping: 0 terms mapped
  Auto-mapped: 0
  Needs review: 0
  Manual required: 0


## Step 4 — Export and Review

Export mappings as DataFrames or CSVs for clinical SME review.

In [12]:
# Export schema mapping as a DataFrame (uses project's engine)
schema_df = schema_map.to_dataframe()
print(f"Schema mapping DataFrame type: {type(schema_df).__name__}")
print(schema_df)

Schema mapping DataFrame type: DataFrame
shape: (6, 6)
┌───────────────────┬──────────────┬──────────────┬───────────────────┬────────────┬───────────────┐
│ source_column     ┆ source_table ┆ target_table ┆ target_column     ┆ confidence ┆ status        │
│ ---               ┆ ---          ┆ ---          ┆ ---               ┆ ---        ┆ ---           │
│ str               ┆ str          ┆ str          ┆ str               ┆ f64        ┆ str           │
╞═══════════════════╪══════════════╪══════════════╪═══════════════════╪════════════╪═══════════════╡
│ patient_id        ┆              ┆ person       ┆ person_id         ┆ 0.95       ┆ auto_accepted │
│ full_name         ┆              ┆ observation  ┆ observation_sourc ┆ 0.3        ┆ unmapped      │
│                   ┆              ┆              ┆ e_value           ┆            ┆               │
│ date_of_birth     ┆              ┆ person       ┆ birth_datetime    ┆ 0.95       ┆ auto_accepted │
│ sex               ┆              ┆

In [13]:
# Export concept mapping as a DataFrame
concept_df = concept_map.to_dataframe()
print(f"Concept mapping DataFrame type: {type(concept_df).__name__}")
print(concept_df.head(10))

Concept mapping DataFrame type: DataFrame
shape: (0, 0)
┌┐
╞╡
└┘


In [14]:
# Export to CSV for SME review
schema_map.to_csv("schema_mapping_hospital_cdm.csv")
concept_map.to_csv("concept_mapping_hospital_cdm.csv")
print("Exported to CSV for clinical review.")

Exported to CSV for clinical review.


## Step 5 — Run ETL to Produce Target Tables

In [15]:
etl_result = project.run_etl(
    source,
    output_dir="./output_hospital_cdm",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)

print(f"Tables written: {len(etl_result.tables)}")
print(f"Output directory: ./output_hospital_cdm/")
for t in etl_result.tables:
    print(f"  {t.table_name}.csv ({t.rows_written} rows)")


2026-04-16 00:35:47 [info     ] Reading source                 format=csv path=example_assets/15_custom_standards/patients.csv


2026-04-16 00:35:47 [info     ] Processing table               columns=3 table=person


2026-04-16 00:35:47 [info     ] ETL complete                   duration=0.002255 success=True


2026-04-16 00:35:47 [info     ] project.etl_complete           output_dir=./output_hospital_cdm project='MIMIC → Hospital CDM'


Tables written: 1
Output directory: ./output_hospital_cdm/
  person.csv (5 rows)


## Tips for Writing Good Custom Standards

1. **Source patterns** — Add common aliases for your columns. The more patterns,
   the more columns get matched instantly (confidence ≥ 0.85) without AI.

2. **Embedding descriptions** — Write natural-language phrases, not just field names.
   Include synonyms and context words. These are what the AI model matches against.

3. **Default entity** — Set `default_entity` and `default_field` to catch unmapped
   columns. Without this, they fall back to the first entity's first field.

4. **Foreign keys** — Specify `fk: "entity.field"` to preserve referential
   integrity in the ETL output.

5. **Iterate** — Run the pipeline, check the confidence scores, then add more
   source_patterns for low-confidence matches. Each pattern you add is a
   guaranteed high-confidence match next time.